# File-Based Photometry Fitting With Bandwagon

This notebook mirrors the DR16Q Bandwagon/SVI workflow, but the targets come from a local file instead of the DR16Q catalog.

Two input styles are supported:

- a source list with `source_id`, `ra`, `dec`, and `redshift` columns, which is then sent through Bandwagon/CDS XMatch
- a long-form photometry file with `source_id`, `filter_name`, `flux_mjy`, and `flux_err_mjy`; add `speclite_name` if the filter name is not one of the built-in mappings below

If explicit `ra`/`dec` columns are absent, the notebook tries to parse SDSS-style IDs such as `012254.62+010106.6` as J2000 coordinates. Redshifts are still required for fixed-redshift fitting; add a `redshift` column to the source file or fill `DEFAULT_REDSHIFT` for a temporary test.

In [ ]:
from pathlib import Path
import re
import sys

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.coordinates import SkyCoord
from astropy.table import Table
from tqdm.auto import tqdm

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
jaxsedfit_src = project_root / "src"
bandwagon_src = project_root.parent / "bandwagon" / "src"
for path in (jaxsedfit_src, bandwagon_src):
    if path.is_dir() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

from bandwagon import DEFAULT_CATALOGS, matches_to_photometry, xmatch_catalogs
from jaxsedfit.config import AGNConfig, FilterSet, FitConfig, GalaxyConfig, InferenceConfig, LikelihoodConfig, Observation, PhotometryData
from jaxsedfit.core import JAXSEDFit
from jaxsedfit.filters import load_filter_curves
from jaxsedfit.mplstyle import style_path

plt.style.use(style_path())

## Settings

In [ ]:
# Change this to the file you want to fit.
SOURCE_FILE = project_root / "table_7.csv"

# If SOURCE_FILE already contains long-form photometry, use it directly.
# If False, SOURCE_FILE is treated as a target list and Bandwagon will query GALEX/SDSS/AllWISE.
USE_PRECOMPUTED_PHOTOMETRY = False

# Use a fixed value only for a quick smoke test. Prefer a real per-source redshift column in SOURCE_FILE.
DEFAULT_REDSHIFT = None

SAMPLE_SIZE = 30
RANDOM_SEED = 20260519
SVI_STEPS = 2000
SVI_LEARNING_RATE = 5e-3
MIN_BANDS_TO_FIT = 5
N_FIT = SAMPLE_SIZE
MAX_MAG_ERR = 1.0

FILTER_SPECLITE_NAME = {
    "FUV_galex": "galex-fuv",
    "NUV_galex": "galex-nuv",
    "u_sdss": "sdss2010-u",
    "g_sdss": "sdss2010-g",
    "r_sdss": "sdss2010-r",
    "i_sdss": "sdss2010-i",
    "z_sdss": "sdss2010-z",
    "W1": "wise2010-W1",
    "W2": "wise2010-W2",
    "W3": "wise2010-W3",
    "W4": "wise2010-W4",
}

FILTER_ORDER = {name: i for i, name in enumerate(FILTER_SPECLITE_NAME)}

DEFAULT_CATALOGS

## Load Targets From File

In [ ]:
def read_table(path):
    path = Path(path)
    if path.suffix.lower() in {".ecsv", ".csv", ".txt", ".dat"}:
        return Table.read(path, format="ascii.ecsv" if path.suffix.lower() == ".ecsv" else "ascii.csv")
    return Table.read(path)


def first_column(table, names):
    lower_to_name = {name.lower(): name for name in table.colnames}
    for name in names:
        if name.lower() in lower_to_name:
            return lower_to_name[name.lower()]
    return None


def parse_sdss_coord(value):
    text = str(value).strip().lstrip("J")
    match = re.fullmatch(r"(\d{2})(\d{2})(\d{2}(?:\.\d+)?)([+-])(\d{2})(\d{2})(\d{2}(?:\.\d+)?)", text)
    if match is None:
        raise ValueError(f"Cannot parse coordinates from source_id={value!r}")
    hh, mm, ss, sign, dd, dm, ds = match.groups()
    ra_deg = 15.0 * (float(hh) + float(mm) / 60.0 + float(ss) / 3600.0)
    dec_abs = float(dd) + float(dm) / 60.0 + float(ds) / 3600.0
    dec_deg = dec_abs if sign == "+" else -dec_abs
    return ra_deg, dec_deg


def as_float_array(values, default=np.nan):
    out = []
    for value in values:
        text = str(value).strip()
        if text == "" or text.lower() in {"nan", "none", "--"}:
            out.append(default)
        else:
            out.append(float(text))
    return np.asarray(out, dtype=float)


def safe_float(value, default=np.nan):
    try:
        text = str(value).strip()
        if text == "" or text.lower() in {"nan", "none", "--"}:
            return default
        return float(text)
    except Exception:
        return default


raw_table = read_table(SOURCE_FILE)
print(f"Read {len(raw_table):,} rows from {SOURCE_FILE}")
print("Columns:", raw_table.colnames)
raw_table[:5]

In [ ]:
id_col = first_column(raw_table, ["source_id", "object_id", "id", "name", "sdss_name"])
if id_col is None:
    source_ids_all = np.asarray([f"source_{i:05d}" for i in range(len(raw_table))], dtype=object)
else:
    source_ids_all = np.asarray([str(value).strip() for value in raw_table[id_col]], dtype=object)

ra_col = first_column(raw_table, ["ra", "ra_deg", "raj2000"])
dec_col = first_column(raw_table, ["dec", "dec_deg", "dej2000"])
if ra_col is not None and dec_col is not None:
    ra_all = as_float_array(raw_table[ra_col])
    dec_all = as_float_array(raw_table[dec_col])
else:
    parsed = [parse_sdss_coord(source_id) for source_id in source_ids_all]
    ra_all = np.asarray([item[0] for item in parsed], dtype=float)
    dec_all = np.asarray([item[1] for item in parsed], dtype=float)

redshift_col = first_column(raw_table, ["redshift", "z", "z_spec", "z_dr16q"])
if redshift_col is not None:
    redshift_all = as_float_array(raw_table[redshift_col])
elif DEFAULT_REDSHIFT is not None:
    redshift_all = np.full(len(raw_table), float(DEFAULT_REDSHIFT))
else:
    redshift_all = np.full(len(raw_table), np.nan)

valid_targets = np.isfinite(ra_all) & np.isfinite(dec_all)
rng = np.random.default_rng(RANDOM_SEED)
valid_idx = np.flatnonzero(valid_targets)
sample_idx = rng.choice(valid_idx, size=min(SAMPLE_SIZE, valid_idx.size), replace=False)

source_ids = source_ids_all[sample_idx]
coords = SkyCoord(ra=ra_all[sample_idx] * u.deg, dec=dec_all[sample_idx] * u.deg, frame="icrs")
sample_meta = {
    str(source_id): {
        "source_id": str(source_id),
        "row_index": int(row_index),
        "ra": float(ra_all[row_index]),
        "dec": float(dec_all[row_index]),
        "redshift": float(redshift_all[row_index]),
    }
    for source_id, row_index in zip(source_ids, sample_idx)
}

print(f"Targets available: {valid_targets.sum():,}")
print(f"Sample size: {len(source_ids):,}")
print("First source IDs:", source_ids[:5].tolist())
if not np.all(np.isfinite(redshift_all[sample_idx])):
    print("Warning: at least one sampled target has no redshift. Add a redshift column or set DEFAULT_REDSHIFT before fitting.")

## Build Photometry

In [ ]:
required_phot_cols = {"source_id", "filter_name", "flux_mjy", "flux_err_mjy"}
has_long_form_photometry = required_phot_cols.issubset(set(raw_table.colnames))

if USE_PRECOMPUTED_PHOTOMETRY:
    if not has_long_form_photometry:
        raise ValueError(f"USE_PRECOMPUTED_PHOTOMETRY=True requires columns {sorted(required_phot_cols)}")
    photometry = raw_table.copy()
    if "speclite_name" not in photometry.colnames:
        photometry["speclite_name"] = [FILTER_SPECLITE_NAME.get(str(name), "") for name in photometry["filter_name"]]
    if "psf_fwhm_arcsec" not in photometry.colnames:
        photometry["psf_fwhm_arcsec"] = np.full(len(photometry), np.nan)
    supported = np.asarray([str(value).strip() != "" for value in photometry["speclite_name"]], dtype=bool)
    finite = np.isfinite(as_float_array(photometry["flux_mjy"])) & np.isfinite(as_float_array(photometry["flux_err_mjy"]))
    positive = (as_float_array(photometry["flux_mjy"]) > 0.0) & (as_float_array(photometry["flux_err_mjy"]) > 0.0)
    selected_ids = set(np.asarray(source_ids, dtype=str))
    selected = np.asarray([str(value) in selected_ids for value in photometry["source_id"]], dtype=bool)
    photometry = photometry[supported & finite & positive & selected]
else:
    # This performs CDS XMatch requests for the sampled targets.
    matches = xmatch_catalogs(coords, source_id=source_ids)
    for key, table in matches.items():
        print(f"{key:10s}: {len(table):6d} matched rows; columns={table.colnames[:12]}")
    photometry = matches_to_photometry(matches, max_mag_err=MAX_MAG_ERR)

print(f"Long-form photometry rows: {len(photometry):,}")
if len(photometry):
    filters, filter_counts = np.unique(np.asarray(photometry["filter_name"], dtype=str), return_counts=True)
    print("Rows by filter:")
    for name, count in zip(filters, filter_counts):
        print(f"  {name:10s} {count:5d}")
photometry[:12]

In [ ]:
phot_source_ids = np.asarray(photometry["source_id"], dtype=str) if len(photometry) else np.asarray([], dtype=str)
unique_ids, counts = np.unique(phot_source_ids, return_counts=True)
source_band_sets = {
    sid: set(np.asarray(photometry[phot_source_ids == sid]["filter_name"], dtype=str))
    for sid in unique_ids
}


def source_priority(source_id):
    bands = source_band_sets[str(source_id)]
    return ("W4" in bands, "W3" in bands, len(bands))


fit_source_ids = [sid for sid, count in zip(unique_ids, counts) if count >= MIN_BANDS_TO_FIT]
fit_source_ids = [sid for sid in fit_source_ids if np.isfinite(sample_meta.get(str(sid), {}).get("redshift", np.nan))]
fit_source_ids = sorted(fit_source_ids, key=source_priority, reverse=True)[:N_FIT]

print(f"Sources with >= {MIN_BANDS_TO_FIT} bands and redshift: {len(fit_source_ids):,}")
print(f"Will fit: {len(fit_source_ids):,}")
if fit_source_ids:
    first_bands = sorted(source_band_sets[fit_source_ids[0]], key=lambda name: FILTER_ORDER.get(name, 999))
    print("First fit source bands:", first_bands)

if len(counts):
    plt.figure(figsize=(7, 4))
    plt.hist(counts, bins=np.arange(0.5, max(counts) + 1.5, 1.0), color="0.25")
    plt.xlabel("usable photometric bands per source")
    plt.ylabel("number of sources")
    plt.tight_layout()

## Build Fit Configs

In [ ]:
dsps_ssp_fn = project_root / "tempdata.h5"
assert dsps_ssp_fn.is_file(), f"DSPS SSP file not found: {dsps_ssp_fn}"


def rows_for_source(source_id):
    mask = np.asarray(photometry["source_id"], dtype=str) == str(source_id)
    rows = photometry[mask]
    order = np.argsort([FILTER_ORDER.get(str(name), 999) for name in rows["filter_name"]])
    return rows[order]


def build_fit_config(source_id, rows):
    meta = sample_meta[str(source_id)]
    if not np.isfinite(meta["redshift"]):
        raise ValueError(f"Missing redshift for {source_id}. Add a redshift column or set DEFAULT_REDSHIFT.")

    filter_names = [str(name) for name in rows["filter_name"]]
    fluxes = np.asarray(rows["flux_mjy"], dtype=float)
    errors = np.asarray(rows["flux_err_mjy"], dtype=float)
    errors = np.maximum(errors, 0.03 * fluxes)

    psf = []
    if "psf_fwhm_arcsec" in rows.colnames:
        psf = [safe_float(value) for value in rows["psf_fwhm_arcsec"]]
    else:
        psf = [np.nan] * len(rows)

    return FitConfig(
        observation=Observation(
            object_id=str(source_id),
            redshift=float(meta["redshift"]),
            redshift_mode="fixed",
            ra=float(meta["ra"]),
            dec=float(meta["dec"]),
        ),
        photometry=PhotometryData(
            filter_names=filter_names,
            fluxes=fluxes.tolist(),
            errors=errors.tolist(),
            is_upper_limit=[False] * len(rows),
            psf_fwhm_arcsec=psf,
        ),
        filters=FilterSet(curves=load_filter_curves(filter_names)),
        galaxy=GalaxyConfig(dsps_ssp_fn=str(dsps_ssp_fn), n_wave=768),
        agn=AGNConfig(agn_type=1),
        likelihood=LikelihoodConfig(
            systematics_width=0.08,
            variability_uncertainty=True,
            use_host_capture_model=True,
        ),
        inference=InferenceConfig(
            map_steps=SVI_STEPS,
            learning_rate=SVI_LEARNING_RATE,
            seed=RANDOM_SEED,
        ),
        prior_config={
            "log_stellar_mass": {"loc": 10.5, "scale": 1.25},
            "fracAGN_5100": {"loc": 0.75, "scale": 0.2},
            "ebv_gal": {"scale": 0.2},
            "ebv_agn": {"scale": 0.2},
        },
    )


example_rows = rows_for_source(fit_source_ids[0]) if fit_source_ids else None
example_cfg = build_fit_config(fit_source_ids[0], example_rows) if fit_source_ids else None
print(example_cfg)

## Fit With SVI

In [ ]:
partial_output_dir = project_root / "notebook_outputs" / "10_file_photometry_bandwagon_svi"
partial_output_dir.mkdir(parents=True, exist_ok=True)
partial_summary_path = partial_output_dir / "partial_fit_summary.ecsv"
partial_failures_path = partial_output_dir / "partial_failed_fits.ecsv"


def table_rows_as_dicts(path):
    if not path.exists():
        return []
    return [dict(row) for row in Table.read(path)]


def save_partial_fit_tables():
    if fit_summaries:
        Table(rows=fit_summaries).write(partial_summary_path, overwrite=True)
    if failed_fits:
        Table(rows=failed_fits).write(partial_failures_path, overwrite=True)


fit_summaries = table_rows_as_dicts(partial_summary_path)
failed_fits = table_rows_as_dicts(partial_failures_path)
completed_source_ids = {str(row["source_id"]) for row in fit_summaries}
failed_source_ids = {str(row["source_id"]) for row in failed_fits}
attempted_source_ids = completed_source_ids | failed_source_ids
remaining_fit_source_ids = [sid for sid in fit_source_ids if str(sid) not in attempted_source_ids]
fitters = []  # collect successful fitter objects for later inspection/plotting
example_fitter = None

if attempted_source_ids:
    print(
        f"Resuming from partial files: {len(completed_source_ids):,} successes and "
        f"{len(failed_source_ids):,} failures already saved."
    )
print(f"Partial summaries will be written to {partial_summary_path}")
print(f"Partial failures will be written to {partial_failures_path}")

for source_id in tqdm(remaining_fit_source_ids, desc="SVI fits"):
    rows = rows_for_source(source_id)
    cfg = build_fit_config(source_id, rows)
    fitter = JAXSEDFit(cfg)
    try:
        map_result = fitter.fit_map(
            steps=SVI_STEPS,
            learning_rate=SVI_LEARNING_RATE,
            progress_bar=False,
        )
        summary = fitter.summary()
        loss = np.asarray(map_result.get('losses', []), dtype=float)
        fit_summaries.append(
            {
                'source_id': str(source_id),
                'redshift': float(cfg.observation.redshift),
                'n_bands': len(rows),
                'final_loss': float(loss[-1]) if loss.size else np.nan,
                'log_stellar_mass_fit': summary.get('log_stellar_mass_fit', np.nan),
                'fracAGN_5100_median': summary.get('fracAGN_5100_median', np.nan),
                'log_agn_bol_luminosity_median': summary.get('log_agn_bol_luminosity_median', np.nan),
            }
        )
        # store fitter for multi-plotting and keep first example as before
        fitters.append(fitter)
        if example_fitter is None:
            example_fitter = fitter
        save_partial_fit_tables()
    except Exception as exc:
        failed_fits.append({
            'source_id': str(source_id),
            'redshift': float(cfg.observation.redshift),
            'n_bands': len(rows),
            'error': repr(exc),
        })
        save_partial_fit_tables()

save_partial_fit_tables()
fit_summary = Table(rows=fit_summaries)
print(f"Successful fits: {len(fit_summary):,}")
print(f"Failed fits: {len(failed_fits):,}")
fit_summary[:10]

In [ ]:
# Plot all successful fits (one figure per source).
# This will call the existing `plot_sed` helper on each stored fitter.
if not globals().get("fitters"):
    print("No successful fits to plot.")
else:
    for i, fitter in enumerate(fitters, start=1):
        obj = fitter.config.observation.object_id
        print(f"Plotting {i}/{len(fitters)}: {obj}")
        try:
            pred = fitter.predict()
            fig = fitter.plot_sed(show=True)
            import matplotlib.pyplot as _plt
            _plt.show()
        except Exception as exc:
            print("Error plotting {obj}:", exc)

## Inspect One Fit

In [ ]:
phot_wave = np.asarray([flt.effective_wavelength for flt in example_fitter.context.filters], dtype=float)
model_flux = np.asarray(pred["pred_fluxes"][0], dtype=float)

print("filter, eff_wave_A, obs_mJy, err_mJy, model_mJy")
for name, wave, obs, err, model in zip(
    example_fitter.config.photometry.filter_names,
    phot_wave,
    example_fitter.config.photometry.fluxes,
    example_fitter.config.photometry.errors,
    model_flux,
):
    print((name, wave, obs, err, model))

## Save Optional Outputs

In [ ]:
# output_dir = project_root / "notebook_outputs" / "10_file_photometry_bandwagon_svi"
# output_dir.mkdir(parents=True, exist_ok=True)
# photometry.write(output_dir / "file_sample_bandwagon_photometry.ecsv", overwrite=True)
# fit_summary.write(output_dir / "file_sample_svi_summary.ecsv", overwrite=True)

## Notes

- The notebook does not touch `05_dr16q_bandwagon_svi.ipynb`.
- For file-driven Bandwagon matching, provide source coordinates either as `ra`/`dec` columns in degrees or SDSS-style coordinate IDs.
- For fitting, provide a real `redshift` column whenever possible. `DEFAULT_REDSHIFT` is only a quick local test hook.
- If your input file is already long-form photometry, set `USE_PRECOMPUTED_PHOTOMETRY=True` and make sure `filter_name` maps to a supported `speclite_name`.